# Kaggle Notebook 01 — Retrieval Experiments and Ablations

Run this notebook on Kaggle for the quantitative part of the NLP project:

- build PDF chunks
- build BM25 + dense FAISS + hybrid/RRF retrieval
- compare BM25 vs dense vs hybrid vs RRF
- alpha ablation
- top-k ablation
- chunk-size ablation
- optional reranking ablation
- offline Logistic Regression learned fusion

In [ ]:
# Kaggle setup. Run this cell first.
%pip install -q pymupdf rank-bm25 sentence-transformers faiss-cpu scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
import sys

# If the notebook is inside this project folder, this works directly.
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    pass
elif (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif Path("/kaggle/working/final-notebooklm-study-assistant").exists():
    PROJECT_ROOT = Path("/kaggle/working/final-notebooklm-study-assistant")
else:
    # Change this if you upload the project somewhere else on Kaggle.
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
BENCHMARK_PATH = PROJECT_ROOT / "benchmarks" / "real_benchmark.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "kaggle_experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PDF files:", list(DATA_DIR.glob("*.pdf")))
print("BENCHMARK_PATH:", BENCHMARK_PATH)

In [ ]:
from study_assistant.retrieval import StudyIndex
from study_assistant.evaluation import (
    load_benchmark_csv,
    run_method_comparison,
    run_alpha_ablation,
    run_topk_ablation,
    run_chunk_ablation,
    evaluate_retriever,
    save_results,
)
from study_assistant.learned_fusion import train_and_evaluate_learned_fusion

index = StudyIndex()
index.build_from_data_dir(DATA_DIR)
print(f"Indexed {len(index.chunks)} chunks")

In [ ]:
# Quick retrieval sanity check
query = "What is the default hybrid retrieval alpha?"
results = index.retrieve_hybrid(query, k=5)
for r in results:
    print(f"{r.source_marker} | score={r.score:.4f} | {r.chunk.filename} page {r.chunk.page}")
    print(r.text[:500])
    print("-" * 100)

In [ ]:
benchmark = load_benchmark_csv(BENCHMARK_PATH)
print("Number of benchmark questions:", len(benchmark))
benchmark[:2]

In [ ]:
# Experiment 1: BM25 vs dense vs hybrid vs RRF
method_df = run_method_comparison(index, benchmark, k=5, alpha=0.30)
save_results(method_df, OUTPUT_DIR / "method_comparison.csv")
method_df

In [ ]:
# Experiment 2: hybrid alpha ablation
alpha_df = run_alpha_ablation(index, benchmark, alphas=[0.3, 0.5, 0.7], k=5)
save_results(alpha_df, OUTPUT_DIR / "alpha_ablation.csv")
alpha_df

In [ ]:
# Experiment 3: top-k ablation
topk_df = run_topk_ablation(index, benchmark, topks=[3, 5, 8], alpha=0.30)
save_results(topk_df, OUTPUT_DIR / "topk_ablation.csv")
topk_df

In [ ]:
# Experiment 4: chunk size ablation
# This rebuilds the embedding index for each setting, so it may take several minutes.
chunk_df = run_chunk_ablation(
    DATA_DIR,
    benchmark,
    configs=[(300, 50), (450, 80), (700, 100)],
    alpha=0.30,
    k=5,
)
save_results(chunk_df, OUTPUT_DIR / "chunk_ablation.csv")
chunk_df

In [ ]:
# Optional Experiment 5: reranking ablation
# This downloads a cross-encoder model and is slower. Run only if needed.
RUN_RERANKER = False

if RUN_RERANKER:
    rerank_result = evaluate_retriever(index, benchmark, method="hybrid", k=5, alpha=0.30, rerank=True)
    rerank_result
else:
    print("Skipped. Set RUN_RERANKER=True to run reranking ablation.")

In [ ]:
# Experiment 6: offline Learned Fusion Ranker using Logistic Regression
# Requires benchmark labels. This is NOT used by the live Streamlit app.
lf_result = train_and_evaluate_learned_fusion(index, benchmark, test_size=0.3, random_state=42)
print(lf_result.metrics)
lf_result.predictions.to_csv(OUTPUT_DIR / "learned_fusion_predictions.csv", index=False)
lf_result.metrics

In [ ]:
# Collect generated CSV files
list(OUTPUT_DIR.glob("*.csv"))

# Robustness Experiment

Use long lecture file (about 100 pages) ``lectures.pdf`` to evaluate ability of long file reading 

In [ ]:
from pathlib import Path
import shutil, glob

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

# Clear old PDFs, including sample_pipeline.pdf
for p in DATA_DIR.glob("*.pdf"):
    p.unlink()

lecture_candidates = glob.glob("/kaggle/input/**/lectures.pdf", recursive=True)

print("Found lectures.pdf candidates:")
for p in lecture_candidates:
    print(p)

assert lecture_candidates, "Cannot find lectures.pdf. Please upload lectures.pdf as Kaggle input first."

LECTURES_SRC = Path(lecture_candidates[0])
LECTURES_DST = DATA_DIR / "lectures.pdf"

shutil.copy2(LECTURES_SRC, LECTURES_DST)

print("Copied to:", LECTURES_DST)
!ls -lh data